<a href="https://colab.research.google.com/github/MarkosTouf/BDA---Final-Project/blob/BDA-FP-Team3/scripts/part3_spark_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BDA Project Part 3: PySpark Analytics in Google Colab

In [146]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BDA_Project_Part03_Pyspark_Analytics_in_Google_Colab")
    .master("local[*]")
    .getOrCreate()
)

print("Spark Session created for BDA Project Part 3 Spark Analytics.")

Spark Session created for BDA Project Part 3 Spark Analytics.


In [147]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

cleaned_market_data_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("interval", StringType(), True),
    StructField("open_time", TimestampType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("close_time", TimestampType(), True),
    StructField("quote_volume", DoubleType(), True),
    StructField("trade_count", DoubleType(), True),
    StructField("taker_buy_base_volume", DoubleType(), True),
    StructField("taker_buy_quote_volume", DoubleType(), True),
    StructField("price_range", DoubleType(), True),
    StructField("price_change", DoubleType(), True),
    StructField("percent_change", DoubleType(), True),
    StructField("candle_direction", StringType(), True),
])

In [148]:
# Team 3 Task 1

print("---- Team 3 Task 1 ----")
print()

cleaned_market_data_path = "/cleaned_market_data.csv"

cleaned_market_data_df = (
    spark.read
    .option("header", True)
    .schema(cleaned_market_data_schema)
    .csv(cleaned_market_data_path)
)

print(f"Loaded file: {cleaned_market_data_path}")

---- Team 3 Task 1 ----

Loaded file: /cleaned_market_data.csv


In [149]:

cleaned_market_data_df_row_count = cleaned_market_data_df.count()
cleaned_market_data_df_columns = cleaned_market_data_df.columns
cleaned_market_data_df_schema = cleaned_market_data_df.schema

print()
print(f"Number of rows: {cleaned_market_data_df_row_count}")
print(f"Cleaned marked data columns: {", ".join(cleaned_market_data_df_columns)}")

print()
print("Cleaned market data schema below:")
cleaned_market_data_df.printSchema()




Number of rows: 8364
Cleaned marked data columns: symbol, interval, open_time, open, high, low, close, volume, close_time, quote_volume, trade_count, taker_buy_base_volume, taker_buy_quote_volume, price_range, price_change, percent_change, candle_direction

Cleaned market data schema below:
root
 |-- symbol: string (nullable = true)
 |-- interval: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- quote_volume: double (nullable = true)
 |-- trade_count: double (nullable = true)
 |-- taker_buy_base_volume: double (nullable = true)
 |-- taker_buy_quote_volume: double (nullable = true)
 |-- price_range: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- percent_change: double (nullable = true)
 |-- candle_direction: str

In [150]:
# Team 3 Task 2

print("---- Team 3 Task 2 ----")
print()

SQL_temp_view_name = "market_data"

cleaned_market_data_df.createOrReplaceTempView(SQL_temp_view_name)

print(f"Created a temporary view: {SQL_temp_view_name}")

# Run SQL Query on Temp View as SQL table name.

print("Running SQL Query on Temp View as SQL table name.")

market_data_rows = spark.sql("""
    SELECT count(1) as row_count
    FROM market_data
""").collect()

print(f"Test SQL query returned row count: {market_data_rows[0]["row_count"]}")

---- Team 3 Task 2 ----

Created a temporary view: market_data
Running SQL Query on Temp View as SQL table name.
Test SQL query returned row count: 8364


In [151]:
# Team 3 Task 3

print("---- Team 3 Task 3 ----")
print()

columns_to_verify = ["price_range", "price_change", "percent_change", "candle_direction"]

columns_to_verify_string = ", ".join(columns_to_verify)

print(f"Verified columns: {columns_to_verify_string}\n")

verify_calculated_row = spark.sql(f"""
    SELECT symbol, {columns_to_verify_string}
    FROM market_data
    LIMIT 5
""").collect()[0]

print("Example row for calculated columns verification:\n")
print(f"symbol={verify_calculated_row["symbol"]} price_range={verify_calculated_row["price_range"]} price_change={verify_calculated_row["price_change"]} percent_change={verify_calculated_row["percent_change"]} candle_direction={verify_calculated_row["candle_direction"]}")


---- Team 3 Task 3 ----

Verified columns: price_range, price_change, percent_change, candle_direction

Example row for calculated columns verification:

symbol=BTCUSDT price_range=190.0 price_change=-22.729999999995925 percent_change=-0.030932109802588266 candle_direction=down


In [152]:
pandas_sample_market_data_path = "/pandas_sample_results.csv"

pandas_sample_market_data_schema = cleaned_market_data_schema

pandas_sample_market_data_df = (
    spark.read
    .option("header", True)
    .schema(pandas_sample_market_data_schema)
    .csv(pandas_sample_market_data_path)
)

print(f"Below step to be used in Team 3 Task 4 comparisons to spark full market data:\n")

print(f"Loaded file for comparisons: {pandas_sample_market_data_path}")

pandas_sample_market_data_df_SQL_View_name = "pandas_sample_market_data"

pandas_sample_market_data_df.createOrReplaceTempView(pandas_sample_market_data_df_SQL_View_name)

print(f"Creating sample pandas data temporary view for comparisons to full spark market data: {pandas_sample_market_data_df_SQL_View_name}")


Below step to be used in Team 3 Task 4 comparisons to spark full market data:

Loaded file for comparisons: /pandas_sample_results.csv
Creating sample pandas data temporary view for comparisons to full spark market data: pandas_sample_market_data


In [153]:
# Team 3 Task 4

print("---- Team 3 Task 4 ----")
print()

print("Team 3 Task 4 part a is enrichment with time related calculated attributes derived from open_time:\n")

from pyspark.sql.functions import date_format, hour, to_date, col

cleaned_market_data_enriched_df = (
    cleaned_market_data_df
    .withColumn("trade_date", to_date(col("open_time")) )
    .withColumn("trade_hour", hour(col("open_time")) )
    .withColumn("day_of_week", date_format(col("open_time"),"E") )

)

time_derived_columns_added = ["trade_date", "trade_hour", "day_of_week"]

print(f"Columns that have been added to enrich dataset are: {", ".join(time_derived_columns_added)}.")
print()

cleaned_market_data_enriched_df_example_row = cleaned_market_data_enriched_df.first()


print(f"Example row after enrichment: trade_date={cleaned_market_data_enriched_df_example_row["trade_date"]}, trade_hour={cleaned_market_data_enriched_df_example_row["trade_hour"]}, day_of_week={cleaned_market_data_enriched_df_example_row["day_of_week"]}")

#

print("Team 3 Task 4 part b is comparison of full enriched market data to pandas sample data:\n")

cleaned_market_data_enriched_df_SQL_View_name = "market_data_time_enriched"

cleaned_market_data_enriched_df.createOrReplaceTempView(cleaned_market_data_enriched_df_SQL_View_name)

print(f"Created full data time enriched temporary view: {cleaned_market_data_enriched_df_SQL_View_name}")

# Run SQL Query on Time Enriched Market Data Temp View as SQL table name.

print("Query 1: Avg Close Price per Symbol")

print("Running SQL Query on Time Enriched full Market Data Temp View as SQL table name.")

print("Running SQL Query 1 - on full cleaned enriched market data - Average close price by symbol:")


# SQL Query 1:

spark.sql("""
    SELECT 'spark full data' as type, symbol, ROUND(AVG(close),2) as avg_close_price
    FROM market_data_time_enriched
    GROUP BY symbol
""").show(truncate=False)


print("Running SQL Query 1 - on pandas sample cleaned market data - Average close price by symbol:")

spark.sql("""
    SELECT 'pandas sample data' as type, symbol, ROUND(AVG(close),2) as avg_close_price
    FROM pandas_sample_market_data
    GROUP BY symbol
""").show(truncate=False)

# Sql Query 2

print("Running SQL Query 2 - on full cleaned enriched market data - Average volume by symbol:")

spark.sql("""
    SELECT 'spark full data' as type, symbol, ROUND(AVG(volume),2) as avg_volume
    FROM market_data_time_enriched
    GROUP BY symbol
""").show(truncate=False)

print("Running SQL Query 2 - on pandas sample cleaned market data - Average volume by symbol:")

spark.sql("""
    SELECT 'pandas sample data' as type, symbol, ROUND(AVG(volume),2) as avg_volume
    FROM pandas_sample_market_data
    GROUP BY symbol
""").show(truncate=False)

# Sql Query 3

print("Running SQL Query 3 - on full cleaned enriched market data - Row Counts by symbol:")

spark.sql("""
    SELECT 'spark full data' as type, symbol, count(1) as row_count
    FROM market_data_time_enriched
    GROUP BY symbol
""").show(truncate=False)

print("Running SQL Query 3 - on pandas sample cleaned market data - Row Counts by symbol:")

spark.sql("""
    SELECT 'pandas sample data' as type, symbol, count(1) as row_count
    FROM pandas_sample_market_data
    GROUP BY symbol
""").show(truncate=False)


---- Team 3 Task 4 ----

Team 3 Task 4 part a is enrichment with time related calculated attributes derived from open_time:

Columns that have been added to enrich dataset are: trade_date, trade_hour, day_of_week.

Example row after enrichment: trade_date=2026-05-29, trade_hour=23, day_of_week=Fri
Team 3 Task 4 part b is comparison of full enriched market data to pandas sample data:

Created full data time enriched temporary view: market_data_time_enriched
Query 1: Avg Close Price per Symbol
Running SQL Query on Time Enriched full Market Data Temp View as SQL table name.
Running SQL Query 1 - on full cleaned enriched market data - Average close price by symbol:
+---------------+--------+---------------+
|type           |symbol  |avg_close_price|
+---------------+--------+---------------+
|spark full data|BTCUSDT |66485.04       |
|spark full data|BNBUSDT |611.41         |
|spark full data|DOGEUSDT|0.09           |
|spark full data|DOTUSDT |1.03           |
|spark full data|LINKUSDT|8.2

Comparisons of Spark full enriched market data to Pandas Sample Market data:

As a first, the comparisons confirm that the number of records used to derive avg_price and avg_volume were different, upwards of 800 in full cleaned market data used in spark analysis and just 5 record samples in pandas sample data.

This proves that the numbers will have sample bias since the stratified sample has relevant samples for reference but does not have the completeness and scope of cleaned data that are available for analysis. Spark is using a more complete cleaned dataset to get more accurate analysis results and is using more datapoints, for more 1 hour interval data from source.

Hence the calculated average prices and volumes calculated in spark analysis with the use of a more complete dataset are more accurate and better capture the true prices and volumes captured from the market.

This highlights the advantage of spark over pandas for data analytics since its capacity of analysing larger datasets and at higher speed gives the capability for higher accuracy and analytics speed and as a result analysis of higher quality and usefulness.

In [154]:
# Team 3 Task 5

print("---- Team 3 Task 5: Volatility Ranking ----")
print()

from pyspark.sql.functions import avg, count, stddev, sum, min, max, round
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window


summary_enriched_for_volatility_ranking_df = cleaned_market_data_enriched_df.groupby("symbol").agg(
    round(avg("price_range"),3).alias("avg_price_range"),
    round(max("price_range"),3).alias("max_price_range"),
    round(min("price_range"),3).alias("min_price_range"),
    round(stddev("price_range"),3).alias("std_dev_price_range"),

)


summary_enriched_for_volatility_ranking_df.show()



---- Team 3 Task 5: Volatility Ranking ----

+--------+---------------+---------------+---------------+-------------------+
|  symbol|avg_price_range|max_price_range|min_price_range|std_dev_price_range|
+--------+---------------+---------------+---------------+-------------------+
| BTCUSDT|        443.724|        3254.17|          59.07|            315.983|
| BNBUSDT|          4.584|          40.99|           0.85|              3.521|
|DOGEUSDT|          0.001|          0.005|            0.0|              0.001|
| DOTUSDT|          0.013|          0.057|          0.003|              0.008|
|LINKUSDT|          0.084|           0.48|          0.014|              0.054|
| SOLUSDT|          0.792|           4.41|           0.19|              0.508|
| ADAUSDT|          0.002|          0.011|            0.0|              0.001|
| XRPUSDT|          0.011|          0.064|          0.003|              0.007|
|AVAXUSDT|          0.086|          0.439|          0.018|              0.056|
| ETHUS

In [155]:
# Adding the window functions for volatility ranking

avg_price_range_window = Window.orderBy(col("avg_price_range").desc())
std_dev_price_range_window = Window.orderBy(col("std_dev_price_range").desc())

ranked_volatility_df = (
  summary_enriched_for_volatility_ranking_df
  .withColumn("std_dev_price_range_rank", dense_rank().over(std_dev_price_range_window))
  .withColumn("avg_price_range_rank", dense_rank().over(avg_price_range_window))


)

# Using price standard deviation for volatility because it is part of definition of market volatility.
# As sec

print("Below Volatility Ranking: ")

ranked_volatility_df.orderBy("std_dev_price_range_rank", "avg_price_range_rank").show()

print("""
Note:
Primary Volatility Ranking measure used as proxy is std dev or price range which is part of market volatility definition.
Secondary volatility ranking measure used is avg price range.
""")


Below Volatility Ranking: 
+--------+---------------+---------------+---------------+-------------------+------------------------+--------------------+
|  symbol|avg_price_range|max_price_range|min_price_range|std_dev_price_range|std_dev_price_range_rank|avg_price_range_rank|
+--------+---------------+---------------+---------------+-------------------+------------------------+--------------------+
| BTCUSDT|        443.724|        3254.17|          59.07|            315.983|                       1|                   1|
| ETHUSDT|         15.547|         108.08|            2.4|             11.864|                       2|                   2|
| BNBUSDT|          4.584|          40.99|           0.85|              3.521|                       3|                   3|
| SOLUSDT|          0.792|           4.41|           0.19|              0.508|                       4|                   4|
|AVAXUSDT|          0.086|          0.439|          0.018|              0.056|                    

In [156]:
# Team 3 Task 6

print("---- Team 3 Task 6: Activity Ranking ----")
print()

print("Task 6 part a: Activity Ranking enrichment with measures:")
print()

summary_enriched_for_activity_ranking_df = cleaned_market_data_enriched_df.groupby("symbol").agg(
    round(sum("trade_count"),2).cast("long").alias("total_trades"),
    round(sum("quote_volume"),2).alias("total_quote_volume"),
    round(avg("volume"),2).alias("average_volume"),

)

summary_enriched_for_activity_ranking_df.show()



---- Team 3 Task 6: Activity Ranking ----

Task 6 part a: Activity Ranking enrichment with measures:

+--------+------------+------------------+--------------+
|  symbol|total_trades|total_quote_volume|average_volume|
+--------+------------+------------------+--------------+
| BTCUSDT|   131613761|     4.28759972E10|        790.22|
| BNBUSDT|    31715734|   3.89696857743E9|       7354.48|
|DOGEUSDT|    21277388|   1.89380720008E9| 2.606019728E7|
| DOTUSDT|     1387241|    2.5093034275E8|     289788.69|
|LINKUSDT|     5451600|    7.0072332144E8|     101243.48|
| SOLUSDT|    32688131|   6.92564896647E9|     113798.94|
| ADAUSDT|     4401497|   1.07006345362E9|    7016408.98|
| XRPUSDT|    24690984|   4.05229693719E9|    4138282.36|
|AVAXUSDT|     8625339|    6.8530839786E8|     115088.05|
| ETHUSDT|   124120341|  2.00681938049E10|      13601.25|
+--------+------------+------------------+--------------+



In [157]:
print("Task 6 part b: Activity Ranking application:")
print()

total_trades_window = Window.orderBy(col("total_trades").desc())
total_quote_volume_window = Window.orderBy(col("total_quote_volume").desc())
average_volume_window = Window.orderBy(col("average_volume").desc())

ranked_activity_df = (
    summary_enriched_for_activity_ranking_df
    .withColumn("total_trades_rank", dense_rank().over(total_trades_window))
    .withColumn("total_quote_volume_rank", dense_rank().over(total_quote_volume_window))
    .withColumn("average_volume_rank", dense_rank().over(average_volume_window))
)

ranked_activity_df.orderBy("total_trades_rank", "total_quote_volume_rank").show()

print("""
Note:
Primary Activity Ranking measure used as proxy is total_trades attribute.
Secondary volatility ranking measure used is total_quote_volume attribute.
Average Volume rank is used as an additional info measure.
Volume will be lower for higher price symbols but trades and quote volume in USDT show the trades activity.
""")




Task 6 part b: Activity Ranking application:

+--------+------------+------------------+--------------+-----------------+-----------------------+-------------------+
|  symbol|total_trades|total_quote_volume|average_volume|total_trades_rank|total_quote_volume_rank|average_volume_rank|
+--------+------------+------------------+--------------+-----------------+-----------------------+-------------------+
| BTCUSDT|   131613761|     4.28759972E10|        790.22|                1|                      1|                 10|
| ETHUSDT|   124120341|  2.00681938049E10|      13601.25|                2|                      2|                  8|
| SOLUSDT|    32688131|   6.92564896647E9|     113798.94|                3|                      3|                  6|
| BNBUSDT|    31715734|   3.89696857743E9|       7354.48|                4|                      5|                  9|
| XRPUSDT|    24690984|   4.05229693719E9|    4138282.36|                5|                      4|               

In [158]:
# Team 3 Task 7

print("---- Team 3 Task 7: Time Based Activity Analysis by Hour and Date ----")
print()

print("Task 7 part a: Activity Ranking by hour - Adding aggregation - total trades by hour:")
print()

summary_enriched_for_trades_by_hour_df = cleaned_market_data_enriched_df.groupby("trade_hour").agg(
    round(sum("trade_count"),2).cast("long").alias("total_trades"),

)

summary_enriched_for_trades_by_hour_df.show()

print("Task 7 part a: Activity Ranking by date:- Adding aggregation - total quote_volume by date:")
print()

summary_enriched_for_quote_volume_by_date_df = cleaned_market_data_enriched_df.groupby("trade_date").agg(
    round(sum("quote_volume"),2).alias("total_quote_volume"),

)

summary_enriched_for_quote_volume_by_date_df.show()




---- Team 3 Task 7: Time Based Activity Analysis by Hour and Date ----

Task 7 part a: Activity Ranking by hour - Adding aggregation - total trades by hour:

+----------+------------+
|trade_hour|total_trades|
+----------+------------+
|        12|    18422164|
|        22|    14229391|
|         1|    15078412|
|        13|    26004194|
|         6|    13655168|
|        16|    20178795|
|         3|    13720639|
|        20|    12689366|
|         5|    12493058|
|        19|    16999745|
|        15|    26252088|
|         9|    12467154|
|        17|    19790920|
|         4|    13518375|
|         8|    12761672|
|        23|    12776777|
|         7|    13815101|
|        10|    12049699|
|        21|    12522814|
|        11|    12300207|
+----------+------------+
only showing top 20 rows
Task 7 part a: Activity Ranking by date:- Adding aggregation - total quote_volume by date:

+----------+------------------+
|trade_date|total_quote_volume|
+----------+------------------+
|2026

In [159]:

# Apply window for ordering by total trades

total_trades_hour_window = Window.orderBy(col("total_trades").desc())

trades_busiest_hour_df = (
    summary_enriched_for_trades_by_hour_df
    .withColumn("trades_hour_rank", dense_rank().over(total_trades_hour_window))
    .filter(col("trades_hour_rank")==1)

)

trades_busiest_hour_df.show()

trades_busiest_hour_df_row = trades_busiest_hour_df.first()

print(f"""
Busiest hour by total trades is:
busiest hour = {trades_busiest_hour_df_row["trade_hour"]} with total trades = {trades_busiest_hour_df_row["total_trades"]}
""")

# Apply window for ordering by total_quote_volumes

total_quote_volumes_date_window = Window.orderBy(col("total_quote_volume").desc())

quote_volume_busiest_date_df = (
    summary_enriched_for_quote_volume_by_date_df
    .withColumn("quote_volume_date_rank", dense_rank().over(total_quote_volumes_date_window))
    .filter(col("quote_volume_date_rank")==1)

)

quote_volume_busiest_date_df.show()

quote_volume_busiest_date_df_row = quote_volume_busiest_date_df.first()

print(f"""
Busiest date by total quote volume is:
busiest date = {quote_volume_busiest_date_df_row["trade_date"]} with total quote volume = {quote_volume_busiest_date_df_row["total_quote_volume"]}
""")

+----------+------------+----------------+
|trade_hour|total_trades|trades_hour_rank|
+----------+------------+----------------+
|        14|    29143296|               1|
+----------+------------+----------------+


Busiest hour by total trades is:
busiest hour = 14 with total trades = 29143296

+----------+------------------+----------------------+
|trade_date|total_quote_volume|quote_volume_date_rank|
+----------+------------------+----------------------+
|2026-06-05|   5.52459750978E9|                     1|
+----------+------------------+----------------------+


Busiest date by total quote volume is:
busiest date = 2026-06-05 with total quote volume = 5524597509.78



Using the above data analysis, the **busiest business hour** across dataset for crypto total trades is at **14:00 hours** which is logical since it is an hour of high market activity generally in markets, with US market participants starting to increase their daily trading activity while European market participants are already in busy trading activity.

On crypto total volumes traded, **June 5th 202**6 was the **busiest date** according to the dataset. Upon some research this was indeed a very active date in crypto trading volume in part triggered by information published on nonfarm payroll data which was viewed and received by market participants as stronger than expected.



In [160]:
# Team 3 Task 8

from pyspark.sql.functions import count_if, when

print("---- Team 3 Task 8: Final Summary by symbol----")
print()

# Step A - aggregation per symbol calculated columns for final summary

summary_part1_by_symbol_df = cleaned_market_data_enriched_df.groupby("symbol").agg(
  count("*").alias("total_records"),
  round(avg("volume"),2).alias("average_volume"),
  sum("trade_count").cast("long").alias("total_trades"),
  round(avg("percent_change"),2).alias("avg_percent_price_change"),
  round(avg("price_range"),2).alias("avg_price_range"),
  sum(when(col("candle_direction")=="up",1).otherwise(0)).alias("up_candle_count"),
  sum(when(col("candle_direction")=="flat",1).otherwise(0)).alias("flat_candle_count"),
  sum(when(col("candle_direction")=="down",1).otherwise(0)).alias("down_candle_count"),


)

summary_part1_by_symbol_df.show()


---- Team 3 Task 8: Final Summary by symbol----

+--------+-------------+--------------+------------+------------------------+---------------+---------------+-----------------+-----------------+
|  symbol|total_records|average_volume|total_trades|avg_percent_price_change|avg_price_range|up_candle_count|flat_candle_count|down_candle_count|
+--------+-------------+--------------+------------+------------------------+---------------+---------------+-----------------+-----------------+
| BTCUSDT|          832|        790.22|   131613761|                   -0.03|         443.72|            384|                1|              447|
| BNBUSDT|          846|       7354.48|    31715734|                   -0.02|           4.58|            412|                1|              433|
|DOGEUSDT|          823| 2.606019728E7|    21277388|                   -0.04|            0.0|            400|                9|              414|
| DOTUSDT|          830|     289788.69|     1387241|                   -0.0

In [161]:
#Now joining summary new derivations to prior dataframes created for volatility and activity rankings

#First I create for final summary a uniform more composite volatility ranking with 2 attributes
# Instead of more granular analytic used before.
# In prior task focus was to provide additional analytical transparency.
# In current task focus is summarized final summary report.

# Step B - unified volatility ranking

for_final_summary_unified_volatility_ranking_df_window = Window.orderBy(col("std_dev_price_range").desc(), col("avg_price_range").desc())


#Adding unified volatility rank. In task 5 I used two ranks as primary and secondary proxy as a more analytic and granular manner.
#For final summary adding a unified volatility rank for final summary simplicity with the 2 volatility proxies used hierarchically for ranking: std_dev_price_range and avg_price_range

final_summary_enriched_for_joining_volatility_ranking_df = (
  summary_enriched_for_volatility_ranking_df
  .withColumn("volatility_rank", dense_rank().over(for_final_summary_unified_volatility_ranking_df_window))
  .select("symbol", "volatility_rank")
)

# Step C - unified activity ranking

for_final_summary_unified_activity_ranking_df_window = Window.orderBy(col("total_trades").desc(), col("total_quote_volume").desc())

final_summary_enriched_for_joining_activity_ranking_df = (
  summary_enriched_for_activity_ranking_df
  .withColumn("activity_rank", dense_rank().over(for_final_summary_unified_activity_ranking_df_window))
  .select("symbol", "activity_rank")
)

# Second, I create for joining to final summary a unified activity ranking attribute.
# Again with the use of the two proxies used prior but in composite hierarchical manner for summary report instead of transparent breakdown approach of before.

joined_summary_and_unified_rankings_by_symbol_df = (
    summary_part1_by_symbol_df
    .join(final_summary_enriched_for_joining_volatility_ranking_df, on=["symbol"], how="left")
    .join(final_summary_enriched_for_joining_activity_ranking_df, on=["symbol"], how="left")
)

joined_summary_and_unified_rankings_by_symbol_df.show()


+--------+-------------+--------------+------------+------------------------+---------------+---------------+-----------------+-----------------+---------------+-------------+
|  symbol|total_records|average_volume|total_trades|avg_percent_price_change|avg_price_range|up_candle_count|flat_candle_count|down_candle_count|volatility_rank|activity_rank|
+--------+-------------+--------------+------------+------------------------+---------------+---------------+-----------------+-----------------+---------------+-------------+
| BTCUSDT|          832|        790.22|   131613761|                   -0.03|         443.72|            384|                1|              447|              1|            1|
| BNBUSDT|          846|       7354.48|    31715734|                   -0.02|           4.58|            412|                1|              433|              3|            4|
|DOGEUSDT|          823| 2.606019728E7|    21277388|                   -0.04|            0.0|            400|           

In [166]:
print(" ------ Final Summary by Symbol ------")

# Get and print the top activity symbol for final summary

final_summary_top_activity_row = (
    joined_summary_and_unified_rankings_by_symbol_df
    .filter(col("activity_rank")==1)
)

top_activity_symbol = final_summary_top_activity_row.first()["symbol"]

print(f"Symbol with highest activity in dataset is: {top_activity_symbol}")

# Get and print the top volatility symbol for final summary

final_summary_top_volatility_row = (
    joined_summary_and_unified_rankings_by_symbol_df
    .filter(col("volatility_rank")==1)
)

top_volatility_symbol = final_summary_top_volatility_row.first()["symbol"]

print(f"Symbol with highest volatility in dataset is: {top_volatility_symbol}")

print("""
Interpretation:
BTCUSDT symbol has both the highest measures of volatility and activity in dataset in the way we define and measure.
Firstly, it has the highest total trade count which is main proxy for trading activity.
Secondly, it has the highest standard deviation of price range - high minus low price - which used as main proxy for volatility.
Is important to note for its interpretation that standard deviation of BTCUSDT price range is high due to
 specific cryptocurrency possessing very high price value in absolute terms.
Price Change Standard deviation therefore is used as volatility proxy with the note that it is metric of absolute volatility as opposed to relative volatility.
For relative volatility metric, percentage price change standard deviation or average could be option to provide a different aspect of analysis.
""")




 ------ Final Summary by Symbol ------
Symbol with highest activity in dataset is: BTCUSDT
Symbol with highest volatility in dataset is: BTCUSDT
